# Reinforcement Learning with Verifiable Rewards (RLVR)

In Session 15 we changed a model's weights with GRPO, rewarding it for verifiably correct answers. This session zooms into the other half of that loop — the **verifier** — and the data pipeline built around it. No GPU required: we run the RLVR sampling-and-verification loop against an API model, so the focus stays on the part that makes or breaks reinforcement learning on language models: the reward signal itself.

The RLVR loop looks like this:

```text
prompt -> sample N completions -> verify each against a deterministic checker
       -> assign rewards -> keep verified-correct samples as preference data
       -> policy update -> repeat
```

Unlike RLHF, there is no learned reward model and no human labeler in the loop. The reward comes from a *deterministic program* — a math answer checker, a unit-test runner — that either passes a completion or doesn't. That makes the signal cheap, objective, and reproducible. It also makes it a target: any policy trained against a verifier will find and exploit its blind spots, so we will also build reward-hacking detection and an audit trail that records every verifier decision.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how RLVR differs from RLHF, and what makes a reward "verifiable."
- Implement verifiable reward functions for math (exact-answer matching) and code (unit-test execution).
- Run the sample-and-verify loop and interpret group-level accuracy.
- Detect reward-hacking signatures and maintain a verifier audit trail.
- Construct chosen/rejected preference pairs ready for DPO-style training — or reward signals ready for GRPO.

## Table of Contents

- **Breakout Room #1: The Verifiable Reward Loop**
  - Task 1: Environment Setup
  - Task 2: Problems and Answer Extraction
  - Task 3: A Math Reward Function
  - Question #1 and Question #2
  - Task 4: Sample and Verify
- **Breakout Room #2: Reward Hacking, Code Verification, and Preference Data**
  - Task 5: Reward-Hacking Detection and the Audit Trail
  - Question #3
  - Task 6: A Code Verifier
  - Question #4
  - Task 7: Build Preference Pairs
  - Activity #1
- **Conclusion: What We Built, Start to Finish**
- **What Looks Different in Production**

---
# Breakout Room #1
## The Verifiable Reward Loop

We build the core RLVR machinery: a small set of math problems with known answers, a deterministic answer checker, a reward function, and the sample-and-verify loop that turns them into training signal.

## Task 1: Environment Setup

From the `16_RLVR` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

You will need an [OpenAI API key](https://platform.openai.com/api-keys). Enter it below — it is kept in memory for this session only.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

### The Policy

In RL terms, the model we sample from is the **policy**. We deliberately use a small model (`gpt-4.1-nano`) at temperature 1.0: a policy that is *sometimes wrong* is exactly what we want, because the contrast between verified-correct and verified-incorrect samples is where the training signal lives. A policy that never fails produces no gradient — and no preference pairs.

In [5]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-4.1-nano"  # small on purpose: we *want* some wrong answers


def simple_complete(prompt: str, system: str = "", temperature: float = 1.0) -> str:
    """One completion from the policy model."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


print(simple_complete("Reply with exactly: policy online"))

policy online


## Task 2: Problems and Answer Extraction

RLVR only works in domains where correctness can be *checked by a program*. Math word problems with a single numeric answer are the canonical example (this is why GSM8K shows up in every RLVR paper — and in Session 15).

Two conventions make checking reliable:

1. Each problem carries a **ground-truth answer** as a string.
2. The prompt instructs the policy to put its final answer in `\boxed{}` — the same convention GSM8K-style training uses — so extraction is a regex, not a judgment call. If no box is found, we fall back to the last number in the text.

In [6]:
import re
from dataclasses import dataclass


@dataclass
class Problem:
    question: str
    answer: str  # ground truth


problems = [
    Problem("What is 12 * 13?", "156"),
    Problem("If a train covers 60 km in 45 minutes, what is its speed in km/h?", "80"),
    Problem("What is the sum of the first 10 positive integers?", "55"),
    Problem(
        "A store discounts a $250 jacket by 20%, then adds 10% sales tax on the discounted price. "
        "What is the final price in dollars?",
        "220",
    ),
    Problem("How many positive divisors does 360 have?", "24"),
]


def extract_number(text: str) -> str:
    r"""Pull the final answer out of a completion: \boxed{...} first, last number as fallback."""
    boxed = re.search(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed.group(1).strip()
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else ""


assert extract_number(r"Step 1: 12*13 = 156. The answer is \boxed{156}.") == "156"
assert extract_number("So the speed is 80 km/h") == "80"
assert extract_number("no numbers here") == ""
print("extraction OK")

extraction OK


## Task 3: A Math Reward Function

The reward function is where verification becomes training signal:

- **+1.0** if the extracted answer matches the ground truth,
- **−0.1** otherwise.

Note the asymmetry: full credit for success, only a *small* penalty for failure. We also normalize numerically (`"80"` and `"80.0"` should match) — a verifier that fails on formatting technicalities punishes correct reasoning, which is the fastest way to teach a policy the wrong lesson.

In [8]:
class MathRewardFunction:
    """Verifiable reward for math problems: exact answer match against ground truth."""

    correct_reward = 1.0
    incorrect_penalty = -0.1

    def compute(self, response: str, ground_truth: str) -> float:
        prediction = extract_number(response)
        if self._normalize(prediction) == self._normalize(ground_truth):
            return self.correct_reward
        return self.incorrect_penalty

    @staticmethod
    def _normalize(value: str):
        """Compare numerically when possible, so '80' == '80.0'."""
        try:
            return float(value)
        except ValueError:
            return value.strip()


reward_fn = MathRewardFunction()

assert reward_fn.compute(r"The speed is \boxed{80}", "80.0") == 1.0
assert reward_fn.compute(r"The speed is \boxed{81}", "80") == -0.1
assert reward_fn.compute("I cannot solve this.", "80") == -0.1
print("reward function OK")

reward function OK


#### ❓ Question #1

In your own words: what makes a reward "verifiable," and how does RLVR differ from RLHF's learned reward model? Give one task where a verifiable reward exists and one where it fundamentally cannot (and explain why).

##### Answer:

A reward is verifiable when a program can check it. You run the checker, it says pass or fail, and it says the same thing every time you run it. Someone else can run the same checker on the same output and get the same result. That is the main difference from RLHF. In RLHF you collect human rankings of model outputs, train a reward model on those rankings, and then the policy optimizes against that reward model. The reward model is a neural net. It can be wrong. It drifts as the policy moves away from the data it was trained on, and the policy can learn to exploit its mistakes. RLVR throws that whole piece out. The reward comes from code, not from a learned model. That makes it cheap, objective, and reproducible. It also means no labelers. The tradeoff is that RLVR only works where correctness is actually well defined.

One of the tasks were a verifiable reward exist in this notebook (Tasks 2 through 4) is the math problems. Math problems have only one right answer. You extract it and compare it to the ground truth. Same for code, where the unit tests either pass or they do not. Verifable reward can't be used in tasks like  e.g writing an email. There is no ground truth to compare against. Two completely different answers can both be good, and which one is better depends on tone, audience, and taste. No program can decide that, because the thing being measured is a human preference, not a fact. That is exactly the case where you still need RLHF and a learned reward model.

#### ❓ Question #2

The reward is asymmetric: +1.0 for a correct answer but only −0.1 for an incorrect one. Suppose we used −1.0 instead. What behavior might a policy learn during early training, when most of its attempts fail? (Hint: think about a model that discovers it can hedge, refuse, or produce no parseable answer at all.)

##### Answer:
With a −1.0 penalty, attempting a problem may not be worthwhile early in training. For example, if the model answers only one out of ten problems correctly, the expected reward with +1.0 for correct and −0.1 for incorrect is still slightly positive: 0.1(1.0) + 0.9(-0.1) = 0.01 So even a weak model is encouraged to keep trying. With a −1.0 penalty, however, the expected reward becomes −0.8, and attempting an answer only becomes worthwhile once the model is correct more than half the time. Because most early attempts fail, the policy may learn to avoid answering rather than improve its reasoning. It might give shorter, safer responses, attempt only easy problems, hedge, refuse, or avoid producing a parseable final answer. The exact behavior depends on how refusals and unparseable answers are rewarded. If they receive 0 while wrong answers receive −1.0, refusing is the safer strategy. The model can avoid the penalty by saying it cannot answer or by never providing a final number. If unparseable answers receive the same penalty as incorrect ones, as in this notebook, refusal offers no advantage. The broader lesson is that refusing or withholding an answer must not be cheaper than making an honest attempt, or the model may learn avoidance instead of reasoning.

## Task 4: Sample and Verify

Now the heart of RLVR: for each problem, sample a **group** of completions at temperature 1.0 and verify every one.

Sampling groups (rather than one completion per prompt) is not incidental — it is the same structure GRPO consumed in Session 15, where each completion's advantage was computed *relative to its group's average reward*. Here the group serves a second purpose too: correct and incorrect completions of the same prompt become the raw material for preference pairs in Breakout Room #2.

In [10]:
from dataclasses import asdict


@dataclass
class Sample:
    problem: str
    response: str
    extracted: str
    reward: float
    verified_correct: bool


def sample_and_verify(problem: Problem, n_samples: int = 4) -> list[Sample]:
    """Sample a group of completions for one problem and verify each one."""
    group = []
    for _ in range(n_samples):
        response = simple_complete(
            f"Solve step by step. Put the final numeric answer in \\boxed{{}}.\n\n{problem.question}",
            system="You are a careful mathematician. Show your work, then box the final number.",
        )
        extracted = extract_number(response)
        reward = reward_fn.compute(response, problem.answer)
        group.append(
            Sample(
                problem=problem.question,
                response=response,
                extracted=extracted,
                reward=reward,
                verified_correct=reward > 0,
            )
        )
    return group


groups = [sample_and_verify(p) for p in problems]

total = sum(len(g) for g in groups)
correct = sum(s.verified_correct for g in groups for s in g)
print(f"Verified-correct rate: {correct / total:.0%} ({correct}/{total})\n")
for problem, group in zip(problems, groups):
    print(f"  {sum(s.verified_correct for s in group)}/{len(group)}  {problem.question[:70]}")

Verified-correct rate: 90% (18/20)

  4/4  What is 12 * 13?
  2/4  If a train covers 60 km in 45 minutes, what is its speed in km/h?
  4/4  What is the sum of the first 10 positive integers?
  4/4  A store discounts a $250 jacket by 20%, then adds 10% sales tax on the
  4/4  How many positive divisors does 360 have?


> NOTE: If your verified-correct rate is 100%, the contrast that drives learning is missing — swap in a smaller model, raise the temperature, or add harder problems until some samples fail.

Let's inspect one failure — reading verifier-rejected completions is how you learn what your policy actually gets wrong (arithmetic slips? misread units? unparseable formatting?):

In [11]:
incorrect = [s for g in groups for s in g if not s.verified_correct]
if incorrect:
    sample = incorrect[0]
    print(f"Problem:   {sample.problem}")
    print(f"Extracted: {sample.extracted!r}  (ground truth mismatch, reward {sample.reward})\n")
    print(sample.response)
else:
    print("No incorrect samples this run - try a harder problem or higher temperature.")

Problem:   If a train covers 60 km in 45 minutes, what is its speed in km/h?
Extracted: '80 \\text{ km/h'  (ground truth mismatch, reward -0.1)

Let's analyze the problem step by step.

**Step 1:** Identify what is given:
- Distance traveled, \( d = 60 \text{ km} \)
- Time taken, \( t = 45 \text{ minutes} \)

**Step 2:** Convert the time from minutes to hours, since we want the speed in km/h.

\[
45 \text{ minutes} = \frac{45}{60} \text{ hours} = \frac{3}{4} \text{ hours}
\]

**Step 3:** Use the basic speed formula:

\[
\text{Speed} = \frac{\text{Distance}}{\text{Time}}
\]

Plugging in the values:

\[
\text{Speed} = \frac{60 \text{ km}}{\frac{3}{4} \text{ hours}}
\]

**Step 4:** Simplify the expression:

\[
\text{Speed} = 60 \div \frac{3}{4} = 60 \times \frac{4}{3}
\]

\[
\text{Speed} = \frac{60 \times 4}{3} = \frac{240}{3} = 80
\]

**Final answer:**

\[
\boxed{80 \text{ km/h}}
\]


## Breakout Room #1 Summary

- Verifiable rewards come from deterministic checkers, not human preference or a learned reward model — cheap, objective, reproducible.
- The `\boxed{}` convention plus numeric normalization makes extraction mechanical; a brittle verifier punishes correct reasoning and corrupts the signal.
- Asymmetric rewards (+1.0 / −0.1) keep early training from collapsing into refusal.
- Sampling *groups* of completions per prompt is the same structure GRPO trains on — and it produces the correct/incorrect contrast that preference data needs.

---
# Breakout Room #2
## Reward Hacking, Code Verification, and Preference Data

A verifier is not just a metric — once you train against it, it *is* the objective. This room covers what happens then: policies that exploit the verifier's blind spots, a second verifiable domain (code judged by unit tests), and turning audited verifier output into preference data.

## Task 5: Reward-Hacking Detection and the Audit Trail

Goodhart's law — *"when a measure becomes a target, it ceases to be a good measure"* — is the central operational risk of RLVR. A policy optimized against an exact-match verifier will happily learn to:

- emit a bare boxed answer with no reasoning (guessing is cheap when only the box is checked),
- parrot numbers that appear in the prompt,
- or exploit extraction quirks instead of solving the problem.

Two defenses, both borrowed from how production RLVR systems are reviewed:

1. **Flag hack signatures.** Here we flag verified-correct samples that show *no visible work* — a right answer without reasoning is the classic signature of guessing or leakage.
2. **Log every verifier decision** to an append-only audit trail (`artifacts/verifier.jsonl`). When someone — a teammate, an auditor, a regulator — asks whether your RL run was trained on honest rewards, this file is the answer.

In [12]:
import json
from pathlib import Path

AUDIT_LOG = Path("artifacts/verifier.jsonl")
AUDIT_LOG.parent.mkdir(exist_ok=True)


def looks_like_hack(sample: Sample) -> bool:
    """Flag verified-correct samples that show no work.

    A correct boxed answer with no visible reasoning is the classic hack
    signature: the policy may be guessing, pattern-matching the prompt, or
    exploiting the extractor rather than solving the problem.
    """
    if not sample.verified_correct:
        return False
    work = sample.response.replace(f"\\boxed{{{sample.extracted}}}", "")
    numbers_in_work = re.findall(r"-?\d+\.?\d*", work)
    return len(sample.response.split()) < 20 or len(numbers_in_work) < 2


def audit_record(sample: Sample) -> dict:
    """Append one verifier decision to the audit trail and return it."""
    record = {**asdict(sample), "suspected_hack": looks_like_hack(sample)}
    with AUDIT_LOG.open("a") as f:
        f.write(json.dumps(record) + "\n")
    return record


records = [audit_record(s) for g in groups for s in g]
flagged = sum(r["suspected_hack"] for r in records)

print(f"Audited {len(records)} samples -> {flagged} flagged as hack-suspect")
print(f"Audit trail: {AUDIT_LOG} ({sum(1 for _ in AUDIT_LOG.open())} records total)")

Audited 20 samples -> 0 flagged as hack-suspect
Audit trail: artifacts/verifier.jsonl (20 records total)


#### ❓ Question #3

Our detector flags one signature: "right answer, no visible work." Name **two other ways** a policy could hack a `\boxed{}` exact-match verifier, and for each, describe how you would harden the verifier or the prompt against it. (Session 15's stacked format rewards are one relevant hardening example.)

##### Answer:

The first one is the fallback rule. If there's no box, extract_number just takes the last number in the text. So the model never has to box anything. It can write whatever it wants, put a number at the end, and get graded on that. It can also spray numbers. Mention several candidate values along the way and make sure the likely one lands last. None of that is solving the problem. It's playing the regex.

To harden it I'd drop the fallback. No box means no answer, and no answer gets the penalty. I'd also require exactly one box and reject the sample if there are two or more, so it can't hedge by listing options. That's the stacked format reward idea from Session 15. Small reward for correct format, separate reward for the correct answer, and correctness only gets checked on one unambiguous value. One caution though. My own run shows the extraction going the other way too. The train problem got extracted as 80 \text{ km/h and scored −0.1 even though the model had it right, because the regex stops at the first closing brace. So if you tighten the format you have to fix the extractor at the same time, or you just start punishing correct answers.

The second one is parroting the prompt. Word problems are full of numbers, and sometimes one of them is the answer or very close to it. The model can learn to pick a number out of the question and box it without doing any arithmetic. It'll be right often enough to be worth doing. Two ways to deal with it. On the verifier side, check if the extracted answer appears verbatim in the question and flag it the same way we flag no visible work. On the data side, write problems where the answer isn't one of the given numbers. The divisors of 360 problem already works like that. If the answer can't be copied then copying stops paying.

One more thing. The detector itself is hackable. looks_like_hack lets through anything over 20 words with two or more numbers in it, so the model can just pad. A few filler sentences, a couple of meaningless numbers, and the flag never fires. My run flagged 0 out of 20, which I don't fully trust as a clean bill of health. Anything you train against becomes part of the objective, detectors included. Checking that the numbers in the work actually lead to the boxed answer would be a better test than counting them.

## Task 6: A Code Verifier

Math is one verifiable domain; **code judged by unit tests** is the other workhorse of RLVR. The verifier executes a candidate program against test cases and returns the *fraction that pass* — a graded reward in `[0.0, 1.0]` rather than math's binary match.

We run candidates in a subprocess with a timeout: a program that crashes, hangs, or exits non-zero simply earns no credit for that test case.

> ⚠️ We are executing model-generated code on your machine. For this demo the programs are trivial, but note the design: in production, this verifier runs inside a **sandbox** (container, gVisor, firecracker VM) — never on the host.

In [13]:
import subprocess
import sys


class CodeVerifier:
    """Score generated code by the fraction of test cases it passes."""

    timeout_seconds = 5

    def verify(self, code: str, test_cases: list[dict]) -> float:
        passed = 0
        for tc in test_cases:
            try:
                output = self._run(code, tc.get("input", ""))
                if output.strip() == str(tc["expected"]).strip():
                    passed += 1
            except Exception:
                pass  # crash, timeout, or non-zero exit -> no credit for this case
        return passed / len(test_cases) if test_cases else 0.0

    def _run(self, code: str, input_data: str = "") -> str:
        result = subprocess.run(
            [sys.executable, "-c", code],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=self.timeout_seconds,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return result.stdout


verifier = CodeVerifier()

# Sanity check with hand-written candidates: one correct, one buggy.
tests = [{"input": "3", "expected": "14"}, {"input": "10", "expected": "385"}]
good = "n = int(input()); print(sum(i * i for i in range(1, n + 1)))"
bad = "n = int(input()); print(sum(range(1, n + 1)))"  # sums i, not i^2

assert verifier.verify(good, tests) == 1.0
assert verifier.verify(bad, tests) == 0.0
print("code verifier OK")

code verifier OK


Now close the loop: have the **policy** write the program, and let the verifier score it — the exact reward signal a coding-RLVR run trains on.

In [14]:
CODING_TASK = (
    "Write a Python program that reads a single integer n from standard input "
    "and prints the sum of the squares of the integers from 1 to n (inclusive). "
    "Print only the number. Reply with only the code - no markdown fences, no explanation."
)

code_tests = [
    {"input": "1", "expected": "1"},
    {"input": "3", "expected": "14"},
    {"input": "10", "expected": "385"},
]


def strip_fences(text: str) -> str:
    """Remove markdown code fences if the policy ignores instructions."""
    return re.sub(r"^```(?:python)?\s*\n|\n?```\s*$", "", text.strip())


for i in range(3):
    candidate = strip_fences(simple_complete(CODING_TASK))
    score = verifier.verify(candidate, code_tests)
    print(f"candidate {i + 1}: reward = {score:.2f}")
    print("  " + candidate.replace("\n", "\n  ") + "\n")

candidate 1: reward = 1.00
  n = int(input())
  print(sum(i*i for i in range(1, n+1)))

candidate 2: reward = 1.00
  n = int(input())
  print(sum(i * i for i in range(1, n + 1)))

candidate 3: reward = 1.00
  n = int(input())
  print(sum(i * i for i in range(1, n + 1)))



#### ❓ Question #4

The code verifier returns *fractional* rewards (fraction of tests passed) while the math verifier is binary. What are the benefits and risks of partial credit as a training signal? And concretely: what could a policy-generated program do to a verifier that runs candidates directly on the host, and which parts of that threat does our timeout **not** cover?

##### Answer:

Partial credit gives us a denser and more useful training signal. A program that passes four out of five tests receives more reward than one that passes none. This lets the policy learn from small improvements instead of receiving zero reward for every imperfect solution. It is especially helpful early in training, when fully correct programs may be rare.

The risk is that the policy may learn to optimize for our tests instead of solving the intended task. It could handle only easy or common cases, hard-code visible answers, or exploit gaps in the test suite. If we miss important edge cases, an incomplete solution could still receive a high reward.

Running generated code directly on our host is also dangerous. The program could read, modify, or delete accessible files, inspect environment variables, make network requests, leak data, consume large amounts of memory or disk space, or create many child processes.

Our timeout only limits how long the monitored subprocess can run. It does not restrict filesystem or network access, limit memory or disk usage, prevent damage before the timeout occurs, or reliably stop child processes that outlive their parent. We therefore need an isolated sandbox with resource limits, network restrictions, and disposable storage. A timeout alone is not enough.

## Task 7: Build Preference Pairs

Finally, we turn audited verifier output into training data. Within each group:

- **chosen** = verified-correct samples that were *not* flagged as hack-suspect,
- **rejected** = verified-incorrect samples,

and we take the cross product. The resulting `{prompt, chosen, rejected}` records are exactly the format [DPO-style trainers](https://huggingface.co/docs/trl/dpo_trainer) consume — while GRPO (Session 15) skips the pairing and uses the group rewards directly. Same verifier, two consumers.

Excluding flagged samples matters: a hack-suspect completion used as "chosen" would teach the next policy iteration to hack *more*.

In [15]:
def build_preferences(groups: list[list[Sample]], records: list[dict]) -> list[dict]:
    """Cross verified-correct (unflagged) winners with incorrect losers, per group."""
    flagged_responses = {r["response"] for r in records if r["suspected_hack"]}
    pairs = []
    for group in groups:
        winners = [s for s in group if s.verified_correct and s.response not in flagged_responses]
        losers = [s for s in group if not s.verified_correct]
        pairs.extend(
            {"prompt": winner.problem, "chosen": winner.response, "rejected": loser.response}
            for winner in winners
            for loser in losers
        )
    return pairs


pairs = build_preferences(groups, records)

PREFERENCES = Path("artifacts/preferences.jsonl")
with PREFERENCES.open("w") as f:
    for pair in pairs:
        f.write(json.dumps(pair) + "\n")

print(f"{len(pairs)} preference pairs -> {PREFERENCES}\n")
if pairs:
    example = pairs[0]
    print(f"prompt:   {example['prompt']}")
    print(f"chosen:   {example['chosen'][:120]}...")
    print(f"rejected: {example['rejected'][:120]}...")
else:
    print("No pairs this run - you need at least one correct AND one incorrect sample in the same group.")

4 preference pairs -> artifacts/preferences.jsonl

prompt:   If a train covers 60 km in 45 minutes, what is its speed in km/h?
chosen:   Let's analyze this problem step by step.

1. **Identify the given data:**
   - Distance covered: 60 km
   - Time taken: ...
rejected: Let's analyze the problem step by step.

**Step 1:** Identify what is given:
- Distance traveled, \( d = 60 \text{ km} \...


#### 🏗️ Activity #1: Build Your Own Verifier

Math answers and unit tests are only two verifiable domains. Pick another — for example:

- **JSON schema conformance**: does the completion parse and validate against a schema?
- **SQL correctness**: does a generated query return the same rows as a reference query on a fixture database?
- **Regex/string transformation**: does the output match a deterministic expected transformation of the input?

Then, in the cell below:

1. Implement a reward function for your domain (binary or fractional — justify the choice).
2. Run the sample-and-verify loop over at least 3 prompts with `n_samples >= 3`.
3. Report the verified-correct rate, and note any hack-suspect behavior you observe (and how you'd detect it).

In [ ]:
@dataclass
class DogTask:
    sentence: str
    expected: dict

dog_tasks = [
    DogTask(
        "Biscuit is a 4-year-old beagle living in Porto.",
        {"name": "Biscuit", "age": 4, "breed": "beagle", "city": "Porto"},
    ),
    DogTask(
        "In Helsinki, a dachshund named Nils just turned 7.",
        {"name": "Nils", "age": 7, "breed": "dachshund", "city": "Helsinki"},
    ),
    DogTask(
        "Kofi, a three-year-old greyhound, was adopted by a family in Warsaw.",
        {"name": "Kofi", "age": 3, "breed": "greyhound", "city": "Warsaw"},
    ),
    DogTask(
        "The border collie Maple, aged 11, herds sheep on a farm outside Brasov.",
        {"name": "Maple", "age": 11, "breed": "border collie", "city": "Brasov"},
    ),
    DogTask(
        "Suki is a whippet who lives in Riga.",
        {"name": "Suki", "age": None, "breed": "whippet", "city": "Riga"},
    ),
    DogTask(
        "Pepper, a pug in Trieste, is 18 months old.",
        {"name": "Pepper", "age": 1.5, "breed": "pug", "city": "Trieste"},
    ),
    DogTask(
        "Dublin is a five-year-old vizsla who lives in Cork.",
        {"name": "Dublin", "age": 5, "breed": "vizsla", "city": "Cork"},
    ),
]

DOG_INSTRUCTIONS = (
    'Reply with only a JSON object with exactly the keys "name", "age", "breed" and "city". '
    '"age" is the dog\'s age in years as a number, not a string, and null if the sentence '
    "does not give it. No markdown fences, no explanation."
)

CHECK_WEIGHTS = {
    "parses": 1,
    "is_object": 1,
    "keys_present": 1,
    "keys_exact": 1,
    "types_match": 1,
    "values_match": 6,
}
CHECKS = tuple(CHECK_WEIGHTS)
TOTAL_WEIGHT = sum(CHECK_WEIGHTS.values())

def same_type(candidate, reference) -> bool:
    """Count every number as one type, so 4 and 4.0 both pass.

    Task 3 normalizes "80" against "80.0" for the same reason: a verifier that fails on a
    formatting technicality punishes an answer that was actually right.
    """
    if isinstance(reference, (int, float)):
        return isinstance(candidate, (int, float)) and not isinstance(candidate, bool)
    return type(candidate) is type(reference)


def values_equal(candidate, reference) -> bool:
    """Compare strings without caring about case or surrounding space.

    DOG_INSTRUCTIONS never asks for verbatim casing, so "Border Collie" is a correct answer
    and scoring it wrong would punish the model for something we did not ask for.
    """
    if isinstance(reference, str) and isinstance(candidate, str):
        return candidate.strip().casefold() == reference.strip().casefold()
    return candidate == reference


def parse_json(text: str):
    """Parse a completion as JSON: returns (parsed, needed_fallback)."""
    candidate = re.sub(r"^```(?:json)?\s*\n|\n?```\s*$", "", text.strip())
    try:
        return json.loads(candidate), False
    except json.JSONDecodeError:
        pass
    block = re.search(r"\{.*\}", candidate, re.DOTALL)
    if block:
        try:
            return json.loads(block.group(0)), True
        except json.JSONDecodeError:
            pass
    return None, False


def score_json(text: str, expected: dict):
    """Reward one completion: the weighted fraction of conformance checks it passes.

    Types and values are gated on the expected keys being *present*, not on the key set
    matching exactly, so one stray extra key costs a single check instead of wiping out
    every check below it.
    """
    parsed, needed_fallback = parse_json(text)
    checks = {"parses": parsed is not None, "is_object": isinstance(parsed, dict)}
    checks["keys_present"] = checks["is_object"] and set(expected) <= set(parsed)
    checks["keys_exact"] = checks["keys_present"] and set(parsed) == set(expected)
    checks["types_match"] = checks["keys_present"] and all(
        same_type(parsed[key], value) for key, value in expected.items()
    )
    checks["values_match"] = checks["types_match"] and all(
        values_equal(parsed[key], value) for key, value in expected.items()
    )
    reward = sum(CHECK_WEIGHTS[name] for name, passed in checks.items() if passed) / TOTAL_WEIGHT
    return reward, checks, parsed, needed_fallback


ground_truth = dog_tasks[0].expected
assert score_json(json.dumps(ground_truth), ground_truth)[0] == 1.0
assert score_json('{"age": 4.0, "name": "Biscuit", "breed": "beagle", "city": "Porto"}', ground_truth)[0] == 1.0
assert score_json('```json\n{"name": "Biscuit", "age": 4, "breed": "beagle", "city": "Porto"}\n```', ground_truth)[0] == 1.0
assert score_json('{"name": "Biscuit", "age": "4", "breed": "beagle", "city": "Porto"}', ground_truth)[0] == 4 / TOTAL_WEIGHT
assert score_json('{"name": "Biscuit", "age": 4, "breed": "Beagle", "city": "porto"}', ground_truth)[0] == 1.0
assert score_json("{}", ground_truth)[0] == 2 / TOTAL_WEIGHT
assert score_json("Sorry, I cannot do that.", ground_truth)[0] == 0.0

extra_key = '{"name": "Biscuit", "age": 4, "breed": "beagle", "city": "Porto", "country": "Portugal"}'
invented = '{"name": "Rex", "age": 9, "breed": "poodle", "city": "Lisbon"}'
assert score_json(extra_key, ground_truth)[0] > score_json(invented, ground_truth)[0]

unstated_age = dog_tasks[4].expected
assert score_json('{"name": "Suki", "age": null, "breed": "whippet", "city": "Riga"}', unstated_age)[0] == 1.0
assert score_json('{"name": "Suki", "age": 3, "breed": "whippet", "city": "Riga"}', unstated_age)[0] == 4 / TOTAL_WEIGHT
print("JSON verifier OK")

JSON verifier OK


In [22]:
DOG_AUDIT_LOG = Path("artifacts/verifier_json.jsonl")
DOG_AUDIT_LOG.parent.mkdir(exist_ok=True)
PLACEHOLDERS = {"", "string", "n/a", "na", "none", "null", "unknown", "name", "breed", "city"}


@dataclass
class DogSample:
    sentence: str
    response: str
    parsed: str
    reward: float
    checks: dict
    needed_fallback: bool
    verified_correct: bool
    flags: list[str]

def hack_flags(task: DogTask, parsed, checks: dict, correct_values: int, needed_fallback: bool) -> list[str]:
    """Signatures worth a human look. A flag is a reason to review, not proof of cheating."""
    flags = []
    if needed_fallback:
        flags.append("needed_fallback")
    if checks["is_object"] and correct_values < len(task.expected) / 2:
        flags.append("shape_only_credit")
    if isinstance(parsed, dict):
        text_values = [value.strip() for value in parsed.values() if isinstance(value, str)]
        if any(value.lower() in PLACEHOLDERS for value in text_values):
            flags.append("placeholder_value")
        if any(len(value) > len(task.sentence) // 2 for value in text_values):
            flags.append("echoed_sentence")
    return flags


def sample_and_verify_dogs(task: DogTask, n_samples: int = 3) -> list[DogSample]:
    """Sample a group of completions for one sentence and score every one."""
    group = []
    for _ in range(n_samples):
        response = simple_complete(
            f"{DOG_INSTRUCTIONS}\n\nSentence: {task.sentence}",
            system="You extract structured data. You reply with JSON and nothing else.",
        )
        reward, checks, parsed, needed_fallback = score_json(response, task.expected)
        correct_values = sum(
            isinstance(parsed, dict) and values_equal(parsed.get(key, ...), value)
            for key, value in task.expected.items()
        )
        group.append(
            DogSample(
                sentence=task.sentence,
                response=response,
                parsed=json.dumps(parsed) if parsed is not None else "",
                reward=reward,
                checks=checks,
                needed_fallback=needed_fallback,
                verified_correct=reward == 1.0,
                flags=hack_flags(task, parsed, checks, correct_values, needed_fallback),
            )
        )
    return group

dog_groups = [sample_and_verify_dogs(task) for task in dog_tasks]
dog_samples = [sample for group in dog_groups for sample in group]

with DOG_AUDIT_LOG.open("a") as audit_file:
    for sample in dog_samples:
        audit_file.write(json.dumps(asdict(sample)) + "\n")

conforming = sum(sample.verified_correct for sample in dog_samples)
mean_reward = sum(sample.reward for sample in dog_samples) / len(dog_samples)
print(f"Fully conforming: {conforming}/{len(dog_samples)} ({conforming / len(dog_samples):.0%})")
print(f"Mean reward:      {mean_reward:.2f}\n")

for task, group in zip(dog_tasks, dog_groups):
    group_mean = sum(sample.reward for sample in group) / len(group)
    conforming_in_group = sum(sample.verified_correct for sample in group)
    print(f"  {conforming_in_group}/{len(group)}  mean {group_mean:.2f}  {task.sentence[:62]}")

print("\nPer-check pass rate (which check is actually failing):")
for check_name in CHECKS:
    passed = sum(sample.checks[check_name] for sample in dog_samples)
    print(f"  {check_name:13s} {passed}/{len(dog_samples)}")

flagged = [sample for sample in dog_samples if sample.flags]
print(f"\nHack-suspect: {len(flagged)}/{len(dog_samples)}")
for sample in flagged:
    print(f"  {','.join(sample.flags):40s} reward {sample.reward:.1f}  {sample.parsed[:60]}")

partial_credit = [sample for sample in dog_samples if 0 < sample.reward < 1]
print(f"\nPartial credit: {len(partial_credit)}/{len(dog_samples)}")
for sample in partial_credit[:4]:
    failed = [check_name for check_name in CHECKS if not sample.checks[check_name]]
    print(f"  reward {sample.reward:.1f}  failed {failed}")
    print(f"    {sample.sentence[:70]}")
    print(f"    {sample.parsed[:100]}")

print(f"\nAudit trail: {DOG_AUDIT_LOG} ({sum(1 for _ in DOG_AUDIT_LOG.open())} records total)")

Fully conforming: 20/21 (95%)
Mean reward:      0.97

  3/3  mean 1.00  Biscuit is a 4-year-old beagle living in Porto.
  3/3  mean 1.00  In Helsinki, a dachshund named Nils just turned 7.
  3/3  mean 1.00  Kofi, a three-year-old greyhound, was adopted by a family in W
  3/3  mean 1.00  The border collie Maple, aged 11, herds sheep on a farm outsid
  3/3  mean 1.00  Suki is a whippet who lives in Riga.
  3/3  mean 1.00  Pepper, a pug in Trieste, is 18 months old.
  2/3  mean 0.79  Dublin is a five-year-old vizsla who lives in Cork.

Per-check pass rate (which check is actually failing):
  parses        21/21
  is_object     21/21
  keys_present  21/21
  keys_exact    21/21
  types_match   20/21
  values_match  20/21

Hack-suspect: 0/21

Partial credit: 1/21
  reward 0.4  failed ['types_match', 'values_match']
    Dublin is a five-year-old vizsla who lives in Cork.
    {"name": null, "age": 5, "breed": "vizsla", "city": "Cork"}

Audit trail: artifacts/verifier_json.jsonl (66 records tot

##### Activity #1 notes

The domain is structured extraction. The policy reads a sentence about a dog and returns the facts as JSON, ground truth is the exact dict, and the checker compares against it instead of judging the output. The reward is fractional because failure here is graded. Output that doesn't parse, output with the right shape but wrong types, and output with the right types but wrong values sit at three different distances from correct, and a binary verifier scores all three the same. The six checks run in order, so the reward can't pay out for a later check while the one it depends on is failing. I normalize numbers so 4 and 4.0 both pass, the same reason Task 3 normalizes 80 against 80.0, and I compare strings without case because the instructions never asked for verbatim casing. Punishing either would mean punishing an answer that was right.

My first run used only the four easy sentences and came out 12 out of 12, mean reward 1.00, nothing flagged. That is what the note under Task 4 warns about: pulling four fields out of a plain sentence is easy in a way multi step arithmetic is not, so there were no failures, no spread inside a group, and no preference pairs to build. A verifier everything passes measures nothing. So I added three sentences built to break in specific ways. Suki has no age in the sentence, so the honest answer is null and any number is an invention I can see. Pepper's age is given in months while the schema asks for years. Dublin is a dog named after a city, in a different city, so name and city can be swapped. The run came out 20 out of 21, mean reward 0.97, and the one failure returned null for the name while getting breed, age and city right. The model would not believe a dog is called Dublin. Suki and Pepper both went three for three, so two of my three traps mostly prove that gpt-4.1-nano is better at this than I expected. Thin contrast, but real, and it came from a sentence I wrote on purpose. That failure is also the first time the fractional reward did any work, scoring 0.36 instead of 0 because only one field of four was missing, which is exactly the gradation a binary verifier throws away. Nothing flagged it, and nothing should have. An honest miss on one field is not a hack.

The asserts cover what one run can't, and that is where I found a real bug. My first version gated every check on an exact key match and weighted all five equally, so a response that read the sentence perfectly but added a fifth key scored 0.4, exactly what an empty object scored, while the right four keys with every value invented scored 0.8. The verifier paid twice as much for shape as for content, which is the Goodhart problem from Task 5 turning up in my own code. Types and values now depend on the expected keys being present rather than on the key set matching exactly, so a stray extra key costs one check instead of wiping out the three below it, and values_match carries a weight of six against one for each structural check. A perfect answer scores 1.0, the same answer with an extra key 0.91, inventing every value 0.45, an empty object 0.18. Shape still collects something, because parsing really is part of the task, but it can no longer outrank reading the sentence.

The signatures I flag are shape_only_credit for reward earned with fewer than half the values correct, placeholder_value for filler like unknown or n/a, echoed_sentence for a value long enough to be a copy of the input, and needed_fallback for a response my strict parse rejected and the loose regex rescued. I first wrote shape_only_credit to fire only at zero correct values, but the name is the easiest word in the sentence to copy, so one lucky field switched the flag off while everything else was still invented. Half is the honest threshold. needed_fallback is not the model cheating, it is my verifier being lenient, and I want it visible. Every decision goes to artifacts/verifier_json.jsonl so the flags can be reviewed rather than trusted.

Strip_fences from Task 6 only strips a python fence, and models fence JSON as a json fence, so reusing it sent every fenced reply down the loose regex path. A verifier helper does not transfer between domains for free. And my third sentence originally put Kofi's family in Poland while the schema asked for a city, so a model that correctly refused to call a country a city would have lost a check for being right. I moved that dog to Warsaw. Bad ground truth fails the same way a bad checker does, it just hides better.

## Breakout Room #2 Summary

- Once you train against a verifier, it *is* the objective — Goodhart's law makes reward-hacking detection and an append-only audit trail (`artifacts/verifier.jsonl`) part of the core pipeline, not an afterthought.
- Code verification generalizes the idea: unit tests yield fractional rewards, and executing untrusted generated code demands sandboxing in anything beyond a demo.
- Verified groups become training data two ways: `{prompt, chosen, rejected}` pairs for DPO-style trainers, or raw group rewards for GRPO — with hack-suspect samples excluded so the next policy doesn't learn to cheat.

Where to go next: feed `artifacts/preferences.jsonl` to TRL's [`DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer); plug these reward functions into Session 15's GRPO run; or read how the labs do it at scale — [Tülu 3](https://arxiv.org/abs/2411.15124) (which coined RLVR) and [DeepSeek-R1](https://arxiv.org/abs/2501.12948).

---
## Conclusion: What We Built, Start to Finish

Walking back through the notebook, the full RLVR arc was:

1. **A policy** (Task 1) — a small API model sampled at temperature 1.0, chosen precisely because it is *sometimes wrong*.
2. **A verifiable domain** (Task 2) — math problems with ground-truth answers and a `\boxed{}` convention that makes checking mechanical.
3. **A reward function** (Task 3) — deterministic, normalized, and asymmetric (+1.0 / −0.1) so early failure doesn't teach refusal.
4. **The sampling loop** (Task 4) — *groups* of completions per prompt, the same structure GRPO computes advantages over, and the source of correct/incorrect contrast.
5. **Adversarial thinking** (Task 5) — once you train against a verifier it *is* the objective, so hack detection and an append-only audit trail are part of the pipeline, not an afterthought.
6. **A second domain** (Task 6) — code judged by unit tests, showing rewards can be fractional and that verification can mean *executing untrusted output*.
7. **Training data** (Task 7) — audited verifier decisions became `{prompt, chosen, rejected}` pairs, with hack-suspects excluded so the next policy iteration doesn't learn to cheat.

The single idea underneath all of it: **wherever a deterministic program can check correctness, you can turn cheap inference into training signal** — no human labelers, no learned reward model. The verifier's quality *is* the ceiling on the policy's quality.

## What Looks Different in Production

This notebook is the smallest honest version of RLVR. Scaling it into a real training pipeline changes nearly every component:

**Sandboxed code execution.** Our `CodeVerifier` runs model-generated code in a bare `subprocess` on your machine — fine for a demo, unacceptable in production, where the policy *will* eventually generate code that reads the filesystem, opens sockets, or fork-bombs the host (and our timeout catches none of that). Production verifiers execute candidates in isolated, disposable environments: [Vercel Sandbox](https://vercel.com/docs/vercel-sandbox) is a good example — ephemeral microVMs built to run untrusted, LLM-generated code with CPU/memory limits, network controls, and full teardown after each run. Self-hosted equivalents include gVisor, Firecracker microVMs, or locked-down containers. The rule: the verifier defines the reward, so the verifier's execution environment is a *security boundary*.

**Scale and throughput.** Five problems × 4 samples becomes tens of thousands of prompts × 8–64 samples per RL step. Sequential API calls give way to async/batched sampling against a dedicated inference fleet (vLLM, as in Session 15) — generation throughput, not the policy update, is usually the bottleneck.

**Verifier hardening.** Regex extraction and exact match get replaced by symbolic math checkers (e.g., SymPy-based equivalence), multiple test suites with hidden held-out cases, and stacked format rewards — because at scale, every blind spot *will* be found and exploited.

**Governance.** Our `verifier.jsonl` becomes real infrastructure: versioned datasets, per-run ledgers, flagged-sample review queues, and dashboards tracking reward distributions for drift. When someone asks "was this model trained on honest rewards?", the audit trail is the answer — this is exactly the artifact regulated environments (the origin of this material) require.

**The training loop itself.** The preference pairs here feed an actual policy update — DPO or GRPO — and then the loop *repeats* against the updated policy: sample, verify, update, again. One pass through this notebook is a single iteration of that flywheel.